<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/06-Evaluate_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating a RAG System

In this notebook, we build a RAG pipeline on a set of LLaMA2 articles and then **systematically evaluate its quality** using three complementary metrics:

- **Hit Rate & MRR** — does the retriever surface the right chunks?
- **Faithfulness & Relevancy** — does the generated answer stay grounded in the retrieved context?
- **Correctness** — how accurate is the answer compared to a known reference?

## What You'll Learn
- How to build and index a document store with ChromaDB + LlamaIndex
- How to generate a synthetic evaluation dataset with an LLM
- How to use `RetrieverEvaluator`, `FaithfulnessEvaluator`, `RelevancyEvaluator`, and `CorrectnessEvaluator`
- How `similarity_top_k` affects retrieval and answer quality

## Prerequisites
- Basic familiarity with Python and RAG concepts
- API keys for OpenAI and Google Gemini (added to Colab Secrets)

## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.20 openai==2.21.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.3 \
                llama-index-llms-google-genai==0.8.7 ipywidgets==7.7.1 jedi==0.19.2

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Create a Vector Store


In [ ]:
import chromadb

# Create client and a new collection.
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

## Load the Articles


In [ ]:
import csv

rows = []

# Load the CSV file
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            # Skip header row
            continue
        rows.append(row)

# Total number of rows in the dataset
print(f"Loaded {len(rows)} rows")

# Convert to Document Objects


In [ ]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1],
        metadata={"title": row[0], "url": row[2], "source_name": row[3]},
    )
    for row in rows
]

# By default, node/chunk IDs are set to random UUIDs.
# To ensure the same IDs per run, we manually set them.
for idx, doc in enumerate(documents):
    doc.id_ = f"doc_{idx}"

In [ ]:
print(documents[0])

# Transforming


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.schema import BaseNode
import hashlib


def deterministic_id_func(i: int, doc: BaseNode) -> str:
    """Generate a unique, repeatable identifier for each node."""
    unique_identifier = doc.id_ + str(i)
    hasher = hashlib.sha256()
    hasher.update(unique_identifier.encode("utf-8"))
    return hasher.hexdigest()


text_splitter = TokenTextSplitter(
    separator=" ", chunk_size=512, chunk_overlap=128, id_func=deterministic_id_func
)

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(model="text-embedding-3-small"),
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(documents=documents, show_progress=True)
print(f"Ingested {len(nodes)} nodes")

In [ ]:
print(nodes[0])

# Load Indexes


In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index from the vector store
index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Define a query engine that retrieves related chunks and uses an LLM to formulate the final answer.
from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(model="gemini-2.5-flash", generation_config=config)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
res = query_engine.query("How many parameters does the LLaMA 2 model have?")

In [ ]:
print(res.response)

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text[:200])
    print("Score\t", src.score)
    print("-_" * 20)

# Evaluate the Retrieval Process and Answer Quality

We can evaluate our RAG system with a dataset of questions and associated chunks. Given a question, we can see if the RAG system retrieves the correct chunks of text that can answer the question.

You can generate a synthetic dataset with an LLM such as `gemini-2.5-flash` or create an authentic and manually curated dataset.

Note that a **well-curated dataset will always be a better option**, especially for a specific domain or use case.


In our example, we will generate a synthetic dataset using `gemini-2.5-flash` to keep it simple.

This is the default prompt that the `generate_question_context_pairs` function uses:

```python
DEFAULT_QA_GENERATE_PROMPT_TMPL = """\
Context information is below.

---------------------
{context_str}
---------------------

Given the context information and no prior knowledge,
generate only questions based on the below query.

You are a Teacher/Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided."
"""
```


In [ ]:
from llama_index.core.llms.utils import LLM
from llama_index.core.schema import MetadataMode, TextNode
from tqdm import tqdm
import json
import re
import uuid
import warnings
import time
from typing import Dict, List, Tuple
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

DEFAULT_QA_GENERATE_PROMPT_TMPL = """\
Context information is below.

---------------------
{context_str}
---------------------

Given the context information and no prior knowledge,
generate only questions based on the below query.

You are a Teacher/Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided."
"""


def generate_question_context_pairs(
    nodes: List[TextNode],
    llm: LLM,
    qa_generate_prompt_tmpl: str = DEFAULT_QA_GENERATE_PROMPT_TMPL,
    num_questions_per_chunk: int = 2,
    request_delay: float = 2.0,
) -> EmbeddingQAFinetuneDataset:
    """Generate question-context pairs from a set of nodes, with rate-limit delays."""
    node_dict = {
        node.node_id: node.get_content(metadata_mode=MetadataMode.NONE)
        for node in nodes
    }

    queries = {}
    relevant_docs = {}

    for node_id, text in tqdm(node_dict.items()):
        query = qa_generate_prompt_tmpl.format(
            context_str=text, num_questions_per_chunk=num_questions_per_chunk
        )
        response = llm.complete(query)

        result = str(response).strip().split("\n")
        questions = [
            re.sub(r"^\d+[\).\s]", "", question).strip() for question in result
        ]
        questions = [question for question in questions if len(question) > 0][
            :num_questions_per_chunk
        ]

        num_questions_generated = len(questions)
        if num_questions_generated < num_questions_per_chunk:
            warnings.warn(
                f"Fewer questions generated ({num_questions_generated}) "
                f"than requested ({num_questions_per_chunk})."
            )

        for question in questions:
            question_id = str(uuid.uuid4())
            queries[question_id] = question
            relevant_docs[question_id] = [node_id]

        time.sleep(request_delay)

    return EmbeddingQAFinetuneDataset(
        queries=queries, corpus=node_dict, relevant_docs=relevant_docs
    )


# Use gemini-2.5-flash (free tier) to generate evaluation questions.
# A request delay is added to stay within rate limits.
from llama_index.llms.google_genai import GoogleGenAI

llm_gen = GoogleGenAI(model="gemini-2.5-flash", temperature=0.3)

rag_eval_dataset = generate_question_context_pairs(
    nodes[:25],
    llm=llm_gen,
    num_questions_per_chunk=1,
    request_delay=4,
)

# Save the dataset as a JSON file for later use
rag_eval_dataset.save_json("./rag_eval_dataset.json")

In [ ]:
# Load the dataset from a previously saved JSON file.
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset.json")

### Evaluation for Hit Rate and Mean Reciprocal Rank (MRR)

We will use `RetrieverEvaluator` from LlamaIndex to measure Hit Rate and MRR.

**Hit Rate:**

Think of Hit Rate like a guessing game. Given a question, you try to find the correct answer from a list of options. Hit Rate measures how often the correct answer appears in your top-k guesses. A high Hit Rate means the retriever frequently finds the right document in its first few results.

**Mean Reciprocal Rank (MRR):**

MRR measures how close to the top the correct document ranks, on average. If the correct document is always the first result, MRR = 1. If it's second, MRR = 0.5. If third, MRR = 0.33 — and so on. A higher MRR means the system ranks the right answer nearer to the top.

In summary: **Hit Rate** tells you how often the system gets it right in its top guesses; **MRR** tells you how close to the top the right answer usually is.


In [ ]:
import pandas as pd


def display_results_retriever(name, eval_results):
    """Display retrieval evaluation results as a DataFrame."""
    metric_dicts = [result.metric_vals_dict for result in eval_results]
    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    return pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# Evaluate the retriever with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(
        rag_eval_dataset, workers=32
    )
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

time.sleep(60)

### Evaluation Using Faithfulness and Relevancy Metrics

Here, we evaluate the answers generated by the LLM. Is the answer grounded in the retrieved context? Is it relevant to the question?

An LLM acts as a judge for these evaluations.

**`FaithfulnessEvaluator`**
Evaluates whether the answer is faithful to the retrieved contexts (i.e., checks for hallucinations).

**`RelevancyEvaluator`**
Evaluates whether the retrieved context and the answer are relevant to the user's question.

Now let's see how the `top_k` value affects these two metrics.


In [ ]:
from llama_index.core.evaluation import RelevancyEvaluator, FaithfulnessEvaluator, BatchEvalRunner
from llama_index.llms.openai import OpenAI

# Recreate the index
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_vector_store(vector_store)

# Define an LLM as judge
llm_gpt5 = OpenAI(model="gpt-5", additional_kwargs={"reasoning_effort": "minimal"})
llm_gpt5_mini = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "minimal"})

# Instantiate the faithfulness and relevancy evaluators
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

# Extract questions from the dataset
queries = list(rag_eval_dataset.queries.values())
# Limit to the first 20 questions to save time
batch_eval_queries = queries[:20]

# The batch evaluator runs the evaluation across multiple queries in parallel
runner = BatchEvalRunner(
    {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
    workers=32,
)

# Try different similarity_top_k values to see how retrieval depth affects answer quality
for i in [2, 4, 6, 8, 10]:
    query_engine = index.as_query_engine(similarity_top_k=i, llm=llm_gpt5_mini)

    eval_results = await runner.aevaluate_queries(query_engine, queries=batch_eval_queries)

    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score:.2f}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score:.2f}")
    print("=" * 15)

### Correctness

The `CorrectnessEvaluator` compares the model's response against a known reference answer and assigns a score from 1–5.


In [ ]:
from llama_index.core.evaluation import CorrectnessEvaluator

query = (
    "Can you explain the theory of relativity proposed by Albert Einstein in detail?"
)

reference = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

General relativity, published in 1915, extended these ideas to include the effects of gravity. According to general relativity, gravity is not a force between masses, as described by Newton's theory of gravity, but rather the result of the warping of space and time by mass and energy. Massive objects, such as planets and stars, cause a curvature in spacetime, and smaller objects follow curved paths in response to this curvature. This concept is often illustrated using the analogy of a heavy ball placed on a rubber sheet, causing it to create a depression that other objects (representing smaller masses) naturally move towards.

In essence, general relativity provided a new understanding of gravity, explaining phenomena like the bending of light by gravity (gravitational lensing) and the precession of the orbit of Mercury. It has been confirmed through numerous experiments and observations and has become a fundamental theory in modern physics.
"""

response = """
Certainly! Albert Einstein's theory of relativity consists of two main components: special relativity and general relativity. Special relativity, published in 1905, introduced the concept that the laws of physics are the same for all non-accelerating observers and that the speed of light in a vacuum is a constant, regardless of the motion of the source or observer. It also gave rise to the famous equation E=mc², which relates energy (E) and mass (m).

However, general relativity, published in 1915, extended these ideas to include the effects of magnetism. According to general relativity, gravity is not a force between masses but rather the result of the warping of space and time by magnetic fields generated by massive objects. Massive objects, such as planets and stars, create magnetic fields that cause a curvature in spacetime, and smaller objects follow curved paths in response to this magnetic curvature. This concept is often illustrated using the analogy of a heavy ball placed on a rubber sheet with magnets underneath, causing it to create a depression that other objects (representing smaller masses) naturally move towards due to magnetic attraction.
"""

In [ ]:
evaluator = CorrectnessEvaluator(llm=llm_gpt5)

result = evaluator.evaluate(query=query, response=response, reference=reference)

In [ ]:
print(f"Correctness score: {result.score}")

In [ ]:
print(result.feedback)

## Optional: Interactive Evaluation Explorer

This section lets you interactively run a single evaluation query and inspect the results — useful for debugging or exploring how the evaluators work on custom inputs.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Text input for the user query
query_input = widgets.Textarea(
    value="What is the context window size of LLaMA 2?",
    description="Query:",
    layout=widgets.Layout(width="80%", height="60px"),
)

# Slider for top_k
topk_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="top_k:"
)

# Button to trigger evaluation
run_button = widgets.Button(description="Run Evaluation", button_style="primary")
output_area = widgets.Output()


def run_eval(b):
    with output_area:
        clear_output()
        q = query_input.value.strip()
        k = topk_slider.value
        if not q:
            print("Please enter a query.")
            return

        print(f"Running query (top_k={k}): {q}\n")
        qe = index.as_query_engine(similarity_top_k=k, llm=llm)
        r = qe.query(q)
        print(f"Answer:\n{r.response}\n")

        faith_eval = FaithfulnessEvaluator(llm=llm_gpt5)
        rel_eval = RelevancyEvaluator(llm=llm_gpt5)

        faith_result = faith_eval.evaluate_response(query=q, response=r)
        rel_result = rel_eval.evaluate_response(query=q, response=r)

        print(f"Faithfulness: {'PASS' if faith_result.passing else 'FAIL'}")
        print(f"Relevancy:    {'PASS' if rel_result.passing else 'FAIL'}")
        print(f"\nSource nodes retrieved: {len(r.source_nodes)}")
        for i, src in enumerate(r.source_nodes, 1):
            print(f"  [{i}] {src.metadata.get('title', 'N/A')} (score={src.score:.3f})")


run_button.on_click(run_eval)

display(
    widgets.VBox([
        widgets.Label("Interactive Evaluation Explorer"),
        query_input,
        topk_slider,
        run_button,
        output_area,
    ])
)